In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ==========================================================
# 1. เตรียมข้อมูลจำลอง (15 Features: 5 Informative, 5 Redundant, 5 Noise)
# ==========================================================
X_raw, y = make_classification(
    n_samples=1000, 
    n_features=15, 
    n_informative=5, 
    n_redundant=5, 
    n_classes=2, 
    random_state=42
)

# ตั้งชื่อ คอลัมน์ f_0 ถึง f_14
feature_names = [f"f_{i}" for i in range(15)]
X = pd.DataFrame(X_raw, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# ==========================================================
# 2. Wrapper Method: Recursive Feature Elimination (RFE)
# ==========================================================
# หลักการ: ใช้ Estimator เทรนข้อมูล แล้วค่อยๆ ตัด Feature ที่มี Weight ต่ำสุดออกทีละตัว
rfe_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('feature_selection', RFE(
        estimator=LogisticRegression(random_state=42), # โมเดลที่ใช้ประเมิน Score ของ Feature
        n_features_to_select=5,                        # จำนวน Feature ที่ต้องการคัดไว้
        step=1                                         # ตัดออกทีละ 1 feature ต่อลูป
    )),
    ('classifier', RandomForestClassifier(random_state=42))
])

# เทรน RFE Pipeline
rfe_pipeline.fit(X_train, y_train)
rfe_preds = rfe_pipeline.predict(X_test)

In [ ]:
# ==========================================================
# 3. Embedded Method: SelectFromModel (L1 / Lasso Regularization)
# ==========================================================
# หลักการ: ใช้โมเดลที่มีกลไกคัดเลือกในตัว (เช่น L1 Penalty ที่บีบ Weight เป็น 0)
embedded_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('feature_selection', SelectFromModel(
        estimator=LogisticRegression(penalty='l1', solver='liblinear', C=0.2, random_state=42),
        threshold='mean' # คัดเฉพาะ Feature ที่มี Coefficient สูงกว่าค่าเฉลี่ย
    )),
    ('classifier', RandomForestClassifier(random_state=42))
])

# เทรน Embedded Pipeline
embedded_pipeline.fit(X_train, y_train)
embedded_preds = embedded_pipeline.predict(X_test)

In [ ]:
# ==========================================================
# 4. ฟังก์ชันช่วยตรวจสอบ Features ที่ผ่านการคัดเลือก (Inspection Helper)
# ==========================================================
def inspect_selected_features(pipeline, input_feature_names):
    # ดึงขั้นตอน Feature Selection จาก Pipeline
    selector = pipeline.named_steps['feature_selection']
    
    # get_support() จะคืนค่าเป็น Boolean Array [True, False, True, ...]
    support_mask = selector.get_support()
    selected_features = np.array(input_feature_names)[support_mask]
    
    return selected_features, support_mask

In [ ]:
# ---------------------------------------------------------
# แสดงผลลัพธ์การเปรียบเทียบ
# ---------------------------------------------------------
rfe_selected, _ = inspect_selected_features(rfe_pipeline, feature_names)
embedded_selected, _ = inspect_selected_features(embedded_pipeline, feature_names)

print("==========================================================")
print("📌 1. Wrapper Method (RFE) Results")
print("==========================================================")
print(f"Selected Features ({len(rfe_selected)}): {list(rfe_selected)}")
print(f"Test Accuracy: {accuracy_score(y_test, rfe_preds):.4f}\n")

print("==========================================================")
print("📌 2. Embedded Method (SelectFromModel - L1) Results")
print("==========================================================")
print(f"Selected Features ({len(embedded_selected)}): {list(embedded_selected)}")
print(f"Test Accuracy: {accuracy_score(y_test, embedded_preds):.4f}\n")